In [659]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

base_url = "http://127.0.0.1:8045"
api_key = "sk-5b2137d3a6c74956a519669d86aa4e71"
client = Anthropic(api_key=api_key, base_url=base_url)
# model = "gemini-3-flash"
model = "claude-sonnet-4-5"

In [660]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return " ".join([block.text for block in message.content if block.type == "text"])

In [661]:
# Fire risk assessment prompt
prompt = """
请按以下步骤分析附件中的房产卫星图像：

1. 住宅识别：定位房产中的主要住宅，查找：

* 最大的有屋顶结构
* 典型住宅特征（车道连接、规则几何形状）
* 与其他结构的区别（车库、棚屋、游泳池）

  2.树冠悬垂分析：检查主要住宅附近的所有树木：

* 识别树冠直接悬垂在屋顶任何部分的树木
* 估算被悬垂树枝覆盖的屋顶百分比（0-25%、25-50%、50-75%、75%+）
* 注意特别密集的悬垂区域

3.火灾风险评估：对于任何悬垂的树木，评估：

* 潜在的野火脆弱性（余烬捕获点、通向结构的连续燃料路径）
* 与烟囱、通风口或其他可见屋顶开口的接近程度
* 树枝在野外植被与结构之间形成“桥梁”的区域

4.防御空间识别：评估房产的整体植被结构：

* 识别树木是否连接形成覆盖或靠近房屋的连续树冠
* 注意任何明显的燃料阶梯（可将火势从地面带到树木再到屋顶的植被）

5.火灾风险评级：根据分析，给出 1-4 级的火灾风险评级：

* 评级 1（低风险）：没有树枝悬垂在屋顶上，房屋周围有良好的防御空间
* 评级 2（中等风险）：最小悬垂（<25% 屋顶），树冠之间有一定分离
* 评级 3（高风险）：显著悬垂（25-50% 屋顶），连接的树冠，多个脆弱点
* 评级 4（严重风险）：广泛悬垂（>50% 屋顶），结构附近密集植被

对于上述每项（1-5），请用一句话总结你的发现，最终回答为数字评级。


"""

In [662]:
import base64
messages = []

with open("./feature_of_claude/images/prop1.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # 图像块
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # 文本块
    {
        "type": "text",
        "text": prompt
    }
])
chat(messages, thinking=True)

Message(id='req_vrtx_011CY9Twuy4MJdfjeCAsF4dY', content=[ThinkingBlock(signature=None, thinking='The user is asking me to analyze a satellite image of a property for fire risk assessment. The instructions are in Chinese, and they want me to:\n\n1. Identify the main residence\n2. Analyze tree canopy overhang\n3. Assess fire risk\n4. Identify defensible space\n5. Provide a fire risk rating (1-4)\n\nLet me analyze the image:\n\nLooking at the satellite image, I can see:\n- A main structure (appears to be a house with a tan/brown roof) in the center\n- Dense green vegetation/trees surrounding the property\n- Dark green areas indicating tree canopies\n- The house appears to be significantly surrounded by trees\n\nNow let me go through each step:\n\n1. **Residential identification**: The main residence appears to be the light brown/tan roofed structure in the center of the image. It has a regular geometric shape and is surrounded by trees.\n\n2. **Tree canopy overhang**: Looking at the image

# 房产卫星图像火灾风险分析  

## 1. 住宅识别 
在图像中上部偏左位置可以识别出一栋主要住宅，呈现规则的矩形屋顶轮廓（灰褐色），有疑似车道或空地连接至上方道路，是该图像中唯一可见的主要建筑结构。

## 2. 树冠悬垂分析
住宅屋顶被周围大量成熟树木严重遮挡，树冠从多个方向（尤其是南侧、东侧和西侧）直接延伸覆盖在屋顶上方，**估计屋顶被树冠覆盖的比例达到50-75%**，仅屋顶中央部分小面积可见，几乎所有屋檐和边缘都被浓密枝叶覆盖。 

## 3. 火灾风险评估 
悬垂的树枝形成了从周围野外植被直达屋顶的连续燃料"桥梁"，余烬极易被屋顶边缘和树枝交汇处捕获；由于树冠覆盖过于密集，无法确认烟囱或通风口的位置，但任何此类开口都将处于树冠笼罩之下，**脆弱性极高**。 

## 4. 防御空间识别 
住宅周围几乎没有有效的防御空间——树木从房屋四周紧密连接形成近乎连续的树冠层，地面植被（草地和灌木）也十分茂密，构成了明显的**燃料阶梯**（地面草丛→灌木→树冠→屋顶），房屋整体被植被"淹没"。  
  
## 5. 火灾风险评级  
综合以上分析：屋顶悬垂面积达50-75%，树冠连续且密集，防御空间几乎为零，燃料阶梯完整，多个脆弱点并存。 
---  
# 🔴 火灾风险评级：**4（严重风险）**  
> 该房产屋顶超过50%被密集树冠直接覆盖，住宅四周完全缺乏有效防御空间，连续的植被和燃料阶梯使结构在野火场景中极度脆弱，强烈建议进行大规模树木修剪和植被清理。

In [663]:
import base64
messages = []

with open("./feature_of_claude/images/prop3.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

add_user_message(messages, [
    # 图像块
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes,
        }
    },
    # 文本块
    {
        "type": "text",
        "text": prompt
    }
])
chat(messages, thinking=True)

Message(id='req_vrtx_011CY9TyNhAzRJYVMbZgqG8X', content=[ThinkingBlock(signature=None, thinking="Let me analyze this satellite image of a property step by step:\n\n1. **Residential Identification**: I can see what appears to be a main residence - the structure with the darker (appears to be dark green/black) roof at the top of the image. There's also a white/light colored structure that appears to be under construction or a patio/deck area with what looks like a swimming pool (light blue oval shape) next to it. The main residence is the structure with the dark roof.\n\n2. **Tree Canopy Overhang Analysis**: Looking at the main residence with the dark roof, I can see several darker green areas around it that appear to be trees. There appears to be tree coverage very close to or possibly overhanging the roof, particularly on the left/upper left side of the structure. The overhang appears to cover approximately 25-40% of the roof area.\n\n3. **Fire Risk Assessment**: The trees appear to be

# 房产卫星图像火灾风险分析

## 1. 住宅识别

主要住宅位于图像中上部，是一栋较大的L形或T形结构，带有棕色/深色屋顶，左侧有一条弧形车道连接，旁边有一个白色/浅色的附属结构（可能是露台或车库延伸），东南方向有一个肾形游泳池，右下方远处有一个小型棚屋/附属建筑。

## 2. 树冠悬垂分析

房屋周围分布有多棵成熟大树，主要集中在**右侧（东侧）和左下角**；从图像上看，右上方有一组树木的树冠边缘**接近但未大面积覆盖屋顶**，屋顶北侧（上方）有少量树枝可能轻微悬垂在屋顶边缘。**估算屋顶被树冠覆盖的比例约为 0-25%**，没有特别密集的悬垂区域。

## 3. 火灾风险评估

东侧和北侧的树木虽然靠近房屋，但树冠与屋顶之间尚保持一定距离，未形成明显的直接"桥梁"通道；未观察到树枝直接覆盖烟囱或通风口区域；从野生植被到结构之间没有连续的燃料路径，游泳池和车道在一定程度上起到了隔离带作用。

## 4. 防御空间识别

周围树木**分散分布，彼此之间的树冠并未形成连续的覆盖层**，房屋四周有大面积修剪整齐的草坪（绿色开阔区域），提供了较好的防御空间；不存在明显的燃料阶梯现象——地面为短草坪，树木为成熟乔木，中间灌木层缺失，这有利于阻断火势垂直蔓延。

## 5. 火灾风险评级

房屋拥有良好的防御空间（大面积草坪），树冠悬垂极少（<25%），树木分散未形成连续冠层，但个别大树距离房屋较近，存在轻微风险。

---

# 最终评级：**2（中等风险）**

> 理由：屋顶树冠悬垂极少但非零，东侧和北侧个别成熟树木距房屋较近，尽管大面积草坪提供了良好的防御空间且无连续树冠，仍存在轻微的余烬接触风险，整体略高于低风险但远未达到高风险水平。

In [665]:
with open("./feature_of_claude/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes,
            },
        },
        {"type": "text", "text": "用一句话总结文档"},
    ],
)

chat(messages)

RateLimitError: Error code: 429 - {'type': 'error', 'error': {'id': 'err_retry_exhausted', 'type': 'rate_limit_error', 'message': 'All 2 attempts failed. Last status: 429 Too Many Requests. Error: HTTP 429: {\n  "error": {\n    "code": 429,\n    "message": "You have exhausted your capacity on this model. Your quota will reset after 0s.",\n    "status": "RESOURCE_EXHAUSTED",\n    "details": [\n      {\n        "@type": "type.googleapis.com/google.rpc.ErrorInfo",\n        "reason": "RATE_LIMIT_EXCEEDED",\n        "domain": "cloudcode-pa.googleapis.com",\n        "metadata": {\n          "uiMessage": "true",\n          "model": "claude-sonnet-4-5",\n          "quotaResetDelay": "193.858664ms",\n          "quotaResetTimeStamp": "2026-02-15T07:35:03Z"\n        }\n      },\n      {\n        "@type": "type.googleapis.com/google.rpc.RetryInfo",\n        "retryDelay": "0.193858664s"\n      }\n    ]\n  }\n}\n'}}

这篇文章全面介绍了地球的物理特性、轨道参数、大气环境、地质演化历史以及其作为太阳系内唯一已知孕育生命的行星的独特性

In [ ]:
with open("./feature_of_claude/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "base64",
                "media_type": "application/pdf",
                "data": file_bytes,
            },
            "title": "地球文章",
            "citations": {"enabled": True},
        },
        {
            "type": "text",
            "text": "地球的大气和海洋是如何形成的？",
        },
    ],
)

message = chat(messages)

from IPython.display import display, Markdown
display(Markdown(message.content[0].text))

根据您提供的资料（第4页“形成后”部分），地球的大气和海洋主要是通过以下方式形成的：

1.  **火山活动与排气作用**：地球形成初期，火山活动和地内排气作用（outgassing）释放了气体，构成了早期的大气层。
2.  **水蒸气凝结**：来自这些来源的水蒸气随后凝结成了海洋。
3.  **外部物质补充**：来自小行星、原行星（protoplanets）和彗星的水和冰也进一步增加了海洋的水量。

In [666]:
article_text = """
地球是距离太阳第三颗行星，也是已知唯一孕育生命的天体。这得益于地球是一个海洋世界，是太阳系中唯一维持液态地表水的行星。地球上的几乎所有水都包含在其全球海洋中，覆盖了地球地壳的 70.8%。地球地壳的其余 29.2% 是陆地，其中大部分以大陆地块的形式位于地球的陆地半球内。地球上的大部分土地至少有些湿润并被植被覆盖，而地球极地沙漠的大片冰层所保留的水比地球的地下水、湖泊、河流和大气中的水加起来还要多。地球地壳由缓慢移动的构造板块组成，这些板块相互作用产生山脉、火山和地震。地球有一个液态外核，能够产生磁层，能够偏转大部分破坏性的太阳风和宇宙辐射。
地球拥有一个动态的大气层，它维持着地球的表面条件，并在进入时保护地球免受大多数流星体和紫外线的伤害。它主要由氮和氧组成。水蒸气在大气中广泛存在，形成覆盖地球大部分地区的云层。水蒸气作为温室气体，与大气中的其他温室气体（特别是二氧化碳 CO2）一起，通过捕获太阳光的能量，为液态地表水和水蒸气的持续存在创造了条件。这个过程维持了当前 14.76 °C（58.57 °F）的平均地表温度，在此温度下，水在正常大气压下呈液态。地理区域之间捕获能量量的差异（如赤道地区比极地地区接收更多阳光）驱动大气和洋流，产生具有不同气候区域的全球气候系统，以及一系列天气现象，如降水，使氮等成分得以循环。
地球
蓝色弹珠，阿波罗 17 号，1972 年 12 月
命名
替代名称
世界 · 地球 ·
Terra · Tellus · Gaia ·
大地母亲 · Sol III
形容词 Earthly · Terrestrial · Terran
· Tellurian
符号和
轨道特征
历元 J2000[n 1]
远日点 152 097 597 km
近日点 147 098 450 km[n 2]
半长轴 149 598 023 km[1]
离心率 0.016 7086[1]
轨道周期
365.256 363 004 天[2]
（恒星年）
(1.000 017 420 96 天文单位)
29.7827 km/s[3]
平均轨道
速度
平近点角 358.617°
轨道倾角 7.155°
– 太阳赤道；
地球被塑造成一个周长约 40,000 公里（25,000 英里）的椭球体。它是太阳系中密度最大的行星。在四颗类地行星中，它是最大和质量最大的。地球距离太阳约八光分，围绕太阳运行，需要一年（约 365.25 天）完成一次公转。地球围绕自己的轴旋转，时间略少于一天（约 23 小时 56 分钟）。地球的自转轴相对于其围绕太阳的轨道平面的垂直方向倾斜，从而产生季节。地球被一颗永久自然卫星——月球所环绕，月球在 384,400 公里（238,900 英里）——1.28 光秒——的距离上围绕地球运行，宽度约为地球的四分之一。月球的引力有助于稳定地球的轴，引起潮汐并逐渐减慢地球的自转。潮汐锁定使月球总是以同一面朝向地球。
地球与太阳系中的大多数其他天体一样，大约在 45 亿年前由早期太阳系中的气体和尘埃形成。在地球历史的第一个十亿年中，海洋形成，然后生命在其中发展。生命在全球范围内传播，并一直在改变地球的大气和表面，导致 20 亿年前的大氧化事件。人类在 30 万年前出现在非洲，并已遍布地球的每个大陆。人类依赖地球的生物圈和自然资源来生存，但越来越影响地球的环境。人类目前对地球气候和生物圈的影响是不可持续的，威胁着人类和许多其他生命形式的生计，并导致广泛的物种灭绝。
[23]
词源
现代英语单词 Earth 通过中古英语从古英语名词发展而来，最常拼写为 eorðe。
[24] 它在所有日耳曼语言中都有同源词，从中已重建出原始日耳曼语 *erþō。在其最早的证明中，单词 eorðe 被用来翻译拉丁语 terra 和希腊语 gē 的多种含义：地面、土壤、旱地、人类世界、世界表面（包括海洋）以及地球本身。与罗马的 Terra（或 Tellus）和
1.578 69°
– 不变
平面；[4]
0.000 05°
– J2000 黄道
升交点经度
−11.260 64°
– J2000
黄道[3]
2023-01-04[5]
近日点时间
近日点幅角
114.207 83°[3]
卫星 1，月球
物理特征
平均半径 6 371.0 km[6]
6 378.137 km[7][8]
赤道
半径
极半径 6 356.752 km[9]
扁率 1/298.257 222 101
(ETRS89)[10]
周长 40 075.017 km 赤道[8]
40 007.86 km
子午线[11][n 3]
表面积 510 072 000 km2[12][n 4]
陆地：148 940 000 km2
水域：361 132 000 km2
体积 1.083 21 × 1012 km3[3]
质量 5.972 168 × 1024 kg[13]
平均密度 5.513 g/cm3[3]
表面重力 9.806 65 m/s2[14]
（恰好 1 g0）
0.3307[15]
转动惯量
因子
逃逸速度 11.186 km/s[3]
会合自转
周期
恒星自转
周期
1.0 天
(24h 00 m 00s)
0.997 269 68 天[16]
(23h 56 m 4.100s)
0.4651 km/s[17]
赤道
自转速度
轴倾角 23.439 2811°[2]
反照率 0.434 几何[3]
0.294 邦德[3]
希腊的 Gaia，地球在日耳曼异教中可能是一个拟人化的女神：晚期北欧神话包括 Jörð（'地球'），一个经常被描述为托尔母亲的巨人。
[25]
温度 255 K（−18 °C）
（黑体温度）[18]
表面温度 最小 平均 最大
[n 5]−89.2 °C 14.76 °C 56.7 °C
0.274 μSv/h[22]
历史上，Earth 以小写形式书写。在中古英语早期，它作为"地球"的明确意义开始使用短语 the earth 来表达。到了现代英语早期，名词的大写开始流行，the earth 也被写成 the Earth，特别是在与其他天体一起引用时。最近，这个名字有时简单地称为 Earth，通过与其他行星名称的类比，尽管 earth 和带有 the earth 的形式仍然很常见。[24]
现在房屋风格各不相同：牛津拼写认为小写形式更常见，大写形式是可接受的变体。另一种约定在作为名称出现时将 Earth 大写，例如"Earth's atmosphere"的描述，但在前面有 the 时使用小写，例如"the atmosphere of the earth"。它几乎总是以小写形式出现在口语表达中，例如"what on earth are you doing?"[26]
表面
等效剂量
率
绝对
星等 (H)
−3.99
大气
表面
101.325 kPa（海平面）
压力
按体积
组成
78.08% 氮（干空气）
20.95% 氧（干空气）
≤1% 水蒸气（可变）
0.9340% 氩
0.0415% 二氧化碳
0.00182% 氖
0.00052% 氦
0.00017% 甲烷
0.00011% 氪
0.00006% 氢
来源：[3]
名称 Terra /ˈtɛrə/ TERR-ə 偶尔在科学写作中使用；它也在科幻小说中用于区分人类居住的行星与其他行星，[27] 而在诗歌中 Tellus /ˈtɛləs/ TELL-əs 已被用来表示地球的拟人化。[28] Terra 也是某些罗曼语中行星的名称，这些语言从拉丁语演变而来，如意大利语和葡萄牙语，而在其他罗曼语中，这个词产生了拼写略有不同的名称，如西班牙语 Tierra 和法语 Terre。希腊诗名 Gaia（[ɡâi ̯ .a] 或 [ɡâj.ja]）的拉丁化形式 Gaea（英语：/ˈdʒiː.ə/ DJEE-ə）很少见，尽管由于盖亚假说，替代拼写 Gaia 已变得常见，在这种情况下，其发音是 /ˈɡaɪ.ə/ GYE-ə 而不是更传统的英语 /ˈɡeɪ.ə/ GAY-ə。
[29]
地球行星有许多形容词。单词 earthly 源自 Earth。从拉丁语 Terra 产生了 terran /ˈtɛrən/ TERR-ən，[30] terrestrial /təˈrɛstriəl/ tərr-EHST-ree-əl，[31] 和（通过法语）terrene /təˈriːn/ tə-REEN，[32] 从拉丁语 Tellus 产生了 tellurian /tɛˈlʊəriən/ teh-LUURR-ee-ən[33] 和 telluric。[34]
自然历史
形成
2012 年早期太阳系原行星盘的 artistic impression，地球和其他太阳系天体从中形成
在太阳系中发现的最古老材料可追溯到 4.5682 +0.0002−0.0004 Ga（十亿年）前。[35] 到 4.54 ± 0.04 Ga，原始地球已经形成。[36] 太阳系中的天体与太阳一起形成和演化。理论上，太阳星云通过引力坍缩从分子云中分割出一个体积，开始旋转并变平成星周盘，然后行星与太阳一起从该盘中生长出来。星云包含气体、冰粒和尘埃（包括原始核素）。根据星云理论，星子通过吸积形成，原始地球估计可能需要 70 到 1 亿年才能形成。[37] 月球年龄的估计范围从 4.5 Ga 到明显更年轻。[38] 一个主要假设是，它是由一个名为 Theia 的、质量约为地球 10% 的火星大小天体与地球碰撞后，从地球释放的材料通过吸积形成的。[39] 它以斜击的方式撞击地球，其部分质量与地球合并。[40][41] 大约在 4.0 到 3.8 Ga 之间，晚期重轰炸期间的众多小行星撞击对月球的更大表面环境造成了重大变化，并由此推断，对地球也是如此。[42]
形成后
地球的大气和海洋是由火山活动和排气形成的。
[43] 来自这些来源的水蒸气凝结成海洋，并得到来自小行星、原行星和彗星的水和冰的补充。
[44] 足以填满海洋的水可能自地球形成以来就存在于地球上。[45] 在这个模型中，当新形成的太阳只有其当前亮度的 70% 时，大气温室气体阻止了海洋结冰。
[46] 到 3.5 Ga，地球的磁场已经建立，这有助于防止大气被太阳风剥离。
[47]
随着地球的熔融外层冷却，它形成了第一个固体地壳，据认为其成分为镁铁质。第一个大陆地壳，其成分更偏长英质，是通过这种镁铁质地壳的部分熔融形成的。[49] 在始太古代沉积岩中存在冥古宙年龄的锆石矿物颗粒，表明至少一些长英质地壳早在 4.4 Ga 就存在，仅在地球形成后 140 Ma。[50] 关于这个初始小体积的大陆地壳如何演化到其当前丰度，有两个主要模型：[51]（1）相对稳定增长直到今天，[52] 这得到了全球大陆地壳的放射性测年的支持，以及（2）在太古代期间大陆地壳体积的初始快速增长，形成了大部分
"""

In [667]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "text",
                "media_type": "text/plain",
                "data": article_text,
            },
            "title": "地球文章",
            "citations": {"enabled": True},
        },
        {
            "type": "text",
            "text": "地球的大气和海洋是如何形成的？",
        },
    ],
)

message = chat(messages)
from IPython.display import display, Markdown
display(Markdown(message.content[0].text))

# 地球大气和海洋的形成

这是一个关于地球科学的问题，让我为您详细解释：

## 🌍 地球大气的形成

### 1. **原始大气（45-40亿年前）**
- **来源**：地球形成初期，从太阳星云中捕获的气体
- **成分**：主要是氢（H₂）和氦（He）
- **消失**：由于地球引力不足以留住这些轻元素，加上太阳风的吹拂，原始大气很快散失

### 2. **次生大气（40-25亿年前）**
- **来源**：火山活动释放的气体（火山脱气）
- **成分**：
  - 水蒸气（H₂O）- 占主要部分
  - 二氧化碳（CO₂）
  - 氮气（N₂）
  - 少量甲烷（CH₄）、氨（NH₃）、硫化物
- **特点**：几乎没有氧气，属于还原性大气

### 3. **现代大气（25亿年前至今）**
- **氧气的出现**：蓝藻等光合生物产生氧气
- **大氧化事件**（约24亿年前）：氧气含量急剧上升
- **臭氧层形成**：保护地球生命免受紫外线伤害

## 🌊 地球海洋的形成

### 1. **水的来源（存在争议）**

**主流理论：**
- **内源说**（主导）：水主要来自地球内部的火山脱气
  - 地幔中含有大量水分
  - 火山喷发释放水蒸气

- **外源说**（补充）：彗星和富含水的小行星撞击
  - 提供了部分水资源
  - 可能占总水量的10-50%

### 2. **海洋形成过程**

**第一阶段：水蒸气积累**（45-40亿年前）
- 地球内部热量驱动火山活动
- 大量水蒸气释放到大气中
- 形成厚重的水蒸气层

**第二阶段：大冷却**（40-38亿年前）
- 地球表面温度逐渐下降（降至100°C以下）
- 水蒸气开始凝结
- 形成持续数百万年的"大雨"

**第三阶段：原始海洋形成**（约38亿年前）
- 雨水汇集在低洼地带
- 形成原始海洋
- 最初的海水温度很高（可能50-80°C）

### 3. **海洋的演化**
- **化学成分变化**：最初偏酸性，逐渐变为弱碱性
- **盐度增加**：岩石风化溶解出矿物质
- **体积变化**：持续的火山活动补充水分

## 📊 时间线总结

```
45亿年前 ─── 地球形成
    │
    ├─ 原始大气（H₂, He）散失
    │
40亿年前 ─── 火山脱气开始
    │       释放H₂O, CO₂, N₂
    │
38亿年前 ─── 地表冷却，原始海洋形成
    │
25亿年前 ─── 光合作用产生O₂
    │
24亿年前 ─── 大氧化事件
    │
现在 ──────── 现代大气和海洋
```

## 🔬 关键证据

- **锆石晶体**：显示38亿年前已有液态水
- **沉积岩**：证明古代海洋的存在
- **同位素分析**：追溯水的来源
- **古生物化石**：记录大气氧含量变化

这个过程是地球从一个炽热星球演变为适宜生命居住的蓝色星球的关键步骤！